In [2]:
import pandas as pd
df = pd.read_parquet("../reports/llm_audits/LLM_DEGRADATION_RESULTS.parquet")
print(df.columns.tolist())
print(df.shape)
print(df.dtypes)

['sample_id', 'city', 'variant_type', 'oracle_label', 'extracted_category', 'masked_instruction', 'llm_output_raw', 'resolution_succeeded', 'predicted_lat', 'predicted_lon', 'gold_goal_lat', 'gold_goal_lon', 'distance_m', 'success_250m', 'success_100m']
(21049, 15)
sample_id                 int64
city                     object
variant_type             object
oracle_label             object
extracted_category       object
masked_instruction       object
llm_output_raw           object
resolution_succeeded       bool
predicted_lat           float64
predicted_lon           float64
gold_goal_lat           float64
gold_goal_lon           float64
distance_m              float64
success_250m             object
success_100m             object
dtype: object


In [2]:
import pandas as pd
df = pd.read_parquet("../reports/llm_audits/LLM_ANSWERABILITY_RESULTS_BASE.parquet")
print(df.columns.tolist())
print(df.shape)
print(df['oracle_label'].value_counts())
print(df['predicted_label'].value_counts())

['sample_id', 'city', 'variant_type', 'oracle_label', 'extracted_category', 'masked_instruction', 'llm_output_raw', 'predicted_label', 'is_correct']
(21049, 9)
oracle_label
Contradictory    9030
Ambiguous        8169
Answerable       3850
Name: count, dtype: int64
predicted_label
Answerable       20975
Contradictory       73
Ambiguous            1
Name: count, dtype: int64


Reminder if we have the original instructions already stored in LLM_DEGRADATION_INPUT.parquet alongside the masked variants?


In [1]:
import pandas as pd
df = pd.read_parquet(
    "/vol/joberant_nobck/data/NLP_368307701_2526a/adanassi/"
    "nlp-allocentric-spatial-reasoning/reports/llm_audits/"
    "LLM_DEGRADATION_INPUT.parquet"
)
print(df.columns.tolist())
print(df['variant_type'].value_counts())
print(df.head(2)[['masked_instruction', 'variant_type', 'oracle_label']])

['sample_id', 'city', 'original_text', 'masked_instruction', 'variant_type', 'removed_element', 'extracted_category', 'extracted_direction', 'extracted_noun', 'start_node', 'gold_goal_node', 'gold_goal_lat', 'gold_goal_lon', 'oracle_label', 'reachable_candidate_count']
variant_type
mask_landmark      7163
mask_directions    6943
mask_both          6943
Name: count, dtype: int64
                                  masked_instruction     variant_type  \
0  Can you meet me at the [MASK] on Liberty Stree...    mask_landmark   
1  Can you meet me at the garden on Liberty Stree...  mask_directions   

  oracle_label  
0    Ambiguous  
1    Ambiguous  


In [1]:
import pandas as pd
df = pd.read_parquet('../reports/llm_audits/LLM_DEGRADATION_RESULTS_LARGE.parquet')
mask_out = df[df['resolution_succeeded']][
    'llm_output_raw'].str.strip().isin(['[MASK]', 'The [MASK]', 
                                         '[DIR_MASK]', 'The [DIR_MASK]'])
print(f"Mask token resolutions: {mask_out.sum()}")

Mask token resolutions: 0


In [3]:
import pandas as pd
df = pd.read_parquet("../data/manhattan/manhattan_silver_standard.parquet")
print(df.columns.tolist())
print(df.head(2))

['sample_id', 'rvs_sample_number', 'city', 'instruction', 'oracle_label', 'candidate_count', 'start_node', 'gold_goal_node', 'gold_goal_lat', 'gold_goal_lon', 'extracted_category', 'extracted_noun', 'extracted_direction', 'target_node']
   sample_id  rvs_sample_number       city  \
0          0                316  manhattan   
1          1                140  manhattan   

                                         instruction oracle_label  \
0  Can you meet me at the garden on Liberty Stree...   Answerable   
1  Head northeast to meet me at the cafe on East ...   Answerable   

   candidate_count   start_node gold_goal_node  gold_goal_lat  gold_goal_lon  \
0              122  #5515156304     #697619118      40.710836     -74.013546   
1              256  #2709347118    #2711466517      40.753225     -73.966858   

  extracted_category extracted_noun extracted_direction   target_node  
0             GARDEN         garden                   N  1#4324586590  
1               CAFE           

In [ ]:
import pandas as pd
print(pd.__version__)

2.3.3


Gap 3 — Per-city breakdown for answerability: PARTIAL
The localization results include per-city breakdowns. But our answerability results do not explicitly report per-city accuracy, only per-variant-type. This is a dataset statistics gap, not a compute gap — the data is already in the parquets, so we extract from them using the following code cell:

In [1]:
import pandas as pd
df = pd.read_parquet('../reports/llm_audits/LLM_ANSWERABILITY_RESULTS_OLMO.parquet')
print(df.groupby('city')['is_correct'].mean().round(3))
print(df.groupby(['city','oracle_label'])['is_correct'].mean().round(3))

city
manhattan       0.382
philadelphia    0.366
pittsburgh      0.360
Name: is_correct, dtype: float64
city          oracle_label 
manhattan     Ambiguous        0.930
              Answerable       0.092
              Contradictory    0.002
philadelphia  Ambiguous        0.936
              Answerable       0.101
              Contradictory    0.001
pittsburgh    Ambiguous        0.901
              Answerable       0.120
              Contradictory    0.002
Name: is_correct, dtype: float64
